# Data Exploration & Transformations: Cleaning and Visualization
Steps: (1) load, (2) inspect, (3) clean and transform, (4) histogram, (5) scatter plot, (6) save the cleaned data.

## 1. Setup and upload

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

uploaded = files.upload()          # choose your dataset (.csv, .xlsx)
filename = list(uploaded.keys())[0]

if filename.endswith(".csv"):
    df = pd.read_csv(filename)
else:
    df = pd.read_excel(filename)

print("Loaded:", filename, "| Shape:", df.shape)
df.head()

## 2. Inspect the raw data

In [ ]:
print("Data types:\n", df.dtypes, "\n")
print("Missing values per column:\n", df.isna().sum(), "\n")
print("Duplicate rows:", df.duplicated().sum())
df.describe(include="all").T

## 3. Clean and transform

In [ ]:
raw_rows = len(df)
clean = df.copy()

# 1. Standardize column names
clean.columns = (clean.columns.str.strip()
                              .str.lower()
                              .str.replace(" ", "_"))

# 2. Remove exact duplicate rows
clean = clean.drop_duplicates()

# 3. Convert numeric-looking columns to numbers (bad entries become NaN)
for col in clean.select_dtypes(include="object").columns:
    converted = pd.to_numeric(clean[col], errors="coerce")
    if converted.notna().sum() > 0.8 * clean[col].notna().sum():
        clean[col] = converted

# 4. Fill missing values: median for numeric, mode for categorical
for col in clean.columns:
    if clean[col].isna().any():
        if pd.api.types.is_numeric_dtype(clean[col]):
            clean[col] = clean[col].fillna(clean[col].median())
        else:
            clean[col] = clean[col].fillna(clean[col].mode()[0])

# 5. Flag outliers using the IQR rule (flagged, not deleted)
numeric_cols = clean.select_dtypes(include="number").columns
for col in numeric_cols:
    q1, q3 = clean[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    clean[f"{col}_outlier"] = ~clean[col].between(q1 - 1.5*iqr, q3 + 1.5*iqr)

print(f"Rows before cleaning: {raw_rows}")
print(f"Rows after cleaning:  {len(clean)}")
print("Outliers flagged per column:")
print(clean[[c for c in clean.columns if c.endswith('_outlier')]].sum())

clean.head()

## 4. Histogram
Change `num_col` to the variable you want to examine.

In [ ]:
num_col = numeric_cols[0]

plt.figure(figsize=(8, 5))
plt.hist(clean[num_col], bins=20, color="steelblue", edgecolor="black")
plt.title(f"Histogram of {num_col}")
plt.xlabel(num_col)
plt.ylabel("Frequency")
plt.grid(axis="y", alpha=0.3)
plt.show()

print(clean[num_col].describe())

## 5. Scatter plot with trend line
Change `x_col` and `y_col` to the variables you want to compare.

In [ ]:
x_col = numeric_cols[0]
y_col = numeric_cols[1]

plt.figure(figsize=(8, 5))
plt.scatter(clean[x_col], clean[y_col], alpha=0.6, color="darkorange", edgecolor="black")

m, b = np.polyfit(clean[x_col], clean[y_col], 1)
xs = np.linspace(clean[x_col].min(), clean[x_col].max(), 100)
plt.plot(xs, m*xs + b, color="red", linewidth=2, label="Trend line")

plt.title(f"{y_col} vs. {x_col}")
plt.xlabel(x_col)
plt.ylabel(y_col)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

r = clean[x_col].corr(clean[y_col])
print(f"Pearson correlation (r) between {x_col} and {y_col}: {r:.3f}")

## 6. Save the cleaned data

In [ ]:
clean.to_csv("cleaned_data.csv", index=False)
files.download("cleaned_data.csv")